In [33]:
from google.colab import files
df = files.upload()
df = pd.read_csv("lowcode_ai_demo.csv")

Saving lowcode_ai_demo.csv to lowcode_ai_demo (3).csv


# Task
Prepare the `df` DataFrame for H2O AutoML by performing an initial data overview, adjusting data types for categorical and datetime columns, initializing the H2O cluster, converting `df` to an H2OFrame, and defining 'high_risk_flag' as the target variable and all other relevant columns as features.

## Data Overview and Basic Checks

### Subtask:
Perform initial data checks such as displaying DataFrame information, descriptive statistics, and checking for missing values.


**Reasoning**:
To perform initial data checks, I will display the DataFrame's concise summary using `.info()`, generate descriptive statistics for numerical columns with `.describe()`, and check for missing values using `.isnull().sum()` as requested.



In [34]:
print("\n--- DataFrame Info ---\n")
df.info()

print("\n--- Descriptive Statistics ---\n")
print(df.describe())

print("\n--- Missing Values ---\n")
print(df.isnull().sum())


--- DataFrame Info ---

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       25000 non-null  int64  
 1   customer_id          25000 non-null  int64  
 2   event_time           25000 non-null  object 
 3   channel              25000 non-null  object 
 4   region               25000 non-null  object 
 5   customer_segment     25000 non-null  object 
 6   device_type          25000 non-null  object 
 7   transaction_amount   25000 non-null  float64
 8   txn_count_last_24h   25000 non-null  int64  
 9   txn_count_last_7d    25000 non-null  int64  
 10  avg_amount_last_30d  25000 non-null  float64
 11  velocity_flag        25000 non-null  int64  
 12  high_amount_flag     25000 non-null  int64  
 13  new_device_flag      25000 non-null  int64  
 14  odd_hour_flag        25000 non-null  int64  
 15  high_risk_f

**Reasoning**:
Based on the `df.info()` output, several columns identified as `object` type (channel, region, customer_segment, device_type) and integer columns representing flags (velocity_flag, high_amount_flag, new_device_flag, odd_hour_flag, high_risk_flag) should be converted to the `category` data type. This is important for H2O AutoML to correctly interpret these as categorical features, and for the target variable `high_risk_flag` to be treated as a classification target. Also, `transaction_id` and `customer_id` should be converted to `category` as they are identifiers and not numerical features.



In [35]:
categorical_cols = [
    'channel',
    'region',
    'customer_segment',
    'device_type',
    'velocity_flag',
    'high_amount_flag',
    'new_device_flag',
    'odd_hour_flag',
    'high_risk_flag',
    'transaction_id',
    'customer_id'
]

for col in categorical_cols:
    df[col] = df[col].astype('category')

print("Data types after conversion:")
df.info()

Data types after conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   transaction_id       25000 non-null  category
 1   customer_id          25000 non-null  category
 2   event_time           25000 non-null  object  
 3   channel              25000 non-null  category
 4   region               25000 non-null  category
 5   customer_segment     25000 non-null  category
 6   device_type          25000 non-null  category
 7   transaction_amount   25000 non-null  float64 
 8   txn_count_last_24h   25000 non-null  int64   
 9   txn_count_last_7d    25000 non-null  int64   
 10  avg_amount_last_30d  25000 non-null  float64 
 11  velocity_flag        25000 non-null  category
 12  high_amount_flag     25000 non-null  category
 13  new_device_flag      25000 non-null  category
 14  odd_hour_flag        25000 non-null  cate

**Reasoning**:
To prepare the `event_time` column for H2O AutoML, it should be converted from `datetime64[ns]` to a numerical format. Converting it to a Unix timestamp (seconds since epoch) is a standard approach, as H2O can then interpret it as a continuous numerical feature.



In [36]:
import pandas as pd

df['event_time'] = pd.to_datetime(df['event_time'])
df['event_time_unix'] = df['event_time'].apply(lambda x: x.timestamp())
df = df.drop(columns=['event_time'])

print("Data types after converting datetime to Unix timestamp:")
df.info()

Data types after converting datetime to Unix timestamp:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   transaction_id       25000 non-null  category
 1   customer_id          25000 non-null  category
 2   channel              25000 non-null  category
 3   region               25000 non-null  category
 4   customer_segment     25000 non-null  category
 5   device_type          25000 non-null  category
 6   transaction_amount   25000 non-null  float64 
 7   txn_count_last_24h   25000 non-null  int64   
 8   txn_count_last_7d    25000 non-null  int64   
 9   avg_amount_last_30d  25000 non-null  float64 
 10  velocity_flag        25000 non-null  category
 11  high_amount_flag     25000 non-null  category
 12  new_device_flag      25000 non-null  category
 13  odd_hour_flag        25000 non-null  category
 14  high_risk_flag

**Reasoning**:
To use H2O AutoML, the first step is to initialize the H2O cluster. This command will start an H2O instance, making it ready for data processing and model training.



**Reasoning**:
The previous command failed because the 'h2o' library was not found. To resolve this, the 'h2o' package needs to be installed using pip.



In [37]:
!pip install h2o
print("h2o library installed.")

h2o library installed.


**Reasoning**:
The previous attempt to install `h2o` failed due to a `SyntaxError` because `pip` commands in a Jupyter/IPython environment need to be prefixed with an exclamation mark (`!`). I will correct the command to `!pip install h2o`.



In [38]:
import sys
!{sys.executable} -m pip install h2o
print("h2o library installed.")

h2o library installed.


**Reasoning**:
The 'h2o' library has been successfully installed in the previous step. Now, I need to initialize the H2O cluster to begin working with H2O-specific functionalities like converting a Pandas DataFrame to an H2OFrame.



In [39]:
import h2o
h2o.init()
print("H2O cluster initialized.")

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,53 mins 34 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,2 months and 20 days
H2O_cluster_name:,H2O_from_python_unknownUser_a5cwcz
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.120 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


H2O cluster initialized.


**Reasoning**:
Now that the H2O cluster is initialized, I will convert the Pandas DataFrame `df` into an H2OFrame, which is a prerequisite for using H2O AutoML. I also need to identify the target variable ('high_risk_flag') and the feature columns (all other columns excluding the target) for model training.



In [40]:
h2o_df = h2o.H2OFrame(df)

y = 'high_risk_flag'
h2o_df[y] = h2o_df[y].asfactor() # Convert target to factor
X = h2o_df.col_names
X.remove(y)

print("H2OFrame created.")
print(f"Target variable (y): {y}")
print(f"Feature columns (X): {X[:5]}... (and {len(X)-5} more)")

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
H2OFrame created.
Target variable (y): high_risk_flag
Feature columns (X): ['transaction_id', 'customer_id', 'channel', 'region', 'customer_segment']... (and 10 more)


## Summary:

### Data Analysis Key Findings

*   The initial `df` DataFrame contained 25,000 entries and 16 columns, with no missing values. It included a mix of `int64`, `datetime64[ns]`, `object`, and `float64` data types.
*   Key categorical columns such as 'channel', 'region', 'customer\_segment', 'device\_type', 'velocity\_flag', 'high\_amount\_flag', 'new\_device\_flag', 'odd\_hour\_flag', 'high\_risk\_flag', 'transaction\_id', and 'customer\_id' were successfully converted to the `category` data type.
*   The `event_time` datetime column was converted to a numerical Unix timestamp (`event_time_unix` as `float64`), and the original column was subsequently dropped.
*   The `h2o` library was successfully installed and the H2O cluster was initialized.
*   The Pandas `df` DataFrame was successfully converted into an `h2o_df` H2OFrame, with 'high\_risk\_flag' designated as the target variable and all other relevant columns defined as features.

### Insights or Next Steps

*   The prepared `h2o_df` H2OFrame, with clearly defined target and feature variables, is now ready for direct use in H2O AutoML for model training and evaluation to predict high-risk transactions.
*   Further feature engineering, such as creating time-based features from the Unix timestamp or aggregating data based on `customer_id`, could potentially enhance model performance.


In [ ]:
from h2o.automl import H2OAutoML

# Initialize H2OAutoML
# Set max_models to 10 to limit the number of models for demonstration purposes
# Set a seed for reproducibility

automl_model = H2OAutoML(max_models=10, seed=42)
automl_model.train(x=X, y=y, training_frame=h2o_df)

# Display the AutoML Leaderboard
lb = automl_model.leaderboard
print(lb.head())

AutoML progress: |████████

In [ ]:
best_model = automl_model.leader
print("Best AutoML Model:")
print(best_model)

In [ ]:
# Split data into training and test sets
train, test = h2o_df.split_frame(ratios=[0.8], seed=42)

# Evaluate the best model on the test set
performance = best_model.model_performance(test)

# Print model performance metrics
print("\n--- Model Performance on Test Set ---\n")
print(performance)

# Display standard plots (e.g., ROC, Confusion Matrix)
print("\n--- ROC Curve ---")
performance.plot()

# The performance.plot() method does not support a 'metric' argument for 'pr' directly.
# If a PR curve is explicitly needed, it would require extracting data and plotting manually with matplotlib.

print("\n--- Confusion Matrix ---")
print(performance.confusion_matrix())

In [ ]:
lb = automl_model.leaderboard.as_data_frame()
# Find the first non-StackedEnsemble model in the leaderboard
non_ensemble_models = lb[~lb['model_id'].str.contains('StackedEnsemble')]

if not non_ensemble_models.empty:
    best_non_ensemble_model_id = non_ensemble_models.iloc[0]['model_id']
    best_varimp_model = h2o.get_model(best_non_ensemble_model_id)

    feature_importance = best_varimp_model.varimp(use_pandas=True)

    print("\n--- Feature Importance (from best non-ensemble model) ---\n")
    print(feature_importance.head(10))

    print("\n--- Feature Importance Plot (from best non-ensemble model) ---")
    best_varimp_model.varimp_plot()
else:
    print("No non-StackedEnsemble models found in the leaderboard that provide variable importances.")

In [ ]:
# Generate predictions (including probabilities) for the entire dataset using the best model
predictions = best_model.predict(h2o_df)

# Convert predictions H2OFrame to Pandas DataFrame
predictions_df = predictions.as_data_frame()

# Append the predicted class and probability columns to the original Pandas DataFrame
df['predicted_high_risk_flag'] = predictions_df['predict']
df['probability_0'] = predictions_df['p0']
df['probability_1'] = predictions_df['p1']

print("Predictions generated and appended to the DataFrame.")
print(df.head())

In [ ]:
import pandas as pd

# Define a range of probability thresholds to simulate
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# Initialize a list to store results
results = []

# Simulate flagging at each threshold
for t in thresholds:
    # Count records where probability_1 is greater than or equal to the threshold
    num_flagged = df[df['probability_1'] >= t].shape[0]
    results.append({'Threshold': t, 'Number of Records Flagged': num_flagged})

# Create a DataFrame from the results
threshold_analysis_df = pd.DataFrame(results)

print("\n--- Number of Records Flagged at Different Probability Thresholds ---\n")
print(threshold_analysis_df)

In [ ]:
import numpy as np

# Define thresholds for risk buckets
low_risk_threshold = 0.3  # Probability below this is Low Risk
medium_risk_threshold = 0.6 # Probability between low_risk_threshold and this is Medium Risk

# Assign risk buckets based on probability_1
conditions = [
    df['probability_1'] < low_risk_threshold,
    (df['probability_1'] >= low_risk_threshold) & (df['probability_1'] < medium_risk_threshold),
    df['probability_1'] >= medium_risk_threshold
]
choices = ['Low', 'Medium', 'High']

df['risk_bucket'] = np.select(conditions, choices, default='Unknown')

print("Risk buckets assigned to the DataFrame.")
print(df[['probability_1', 'risk_bucket']].head())
print("\nDistribution of Risk Buckets:")
print(df['risk_bucket'].value_counts())

In [ ]:
output_path = "lowcode_ai_scored_output.csv" # Colab saves to content directory by default
df.to_csv(output_path, index=False)
print(f"DataFrame saved to {output_path}")